# Corrected Supplementary class-wise and synthetic visuals

This standalone notebook regenerates the ten established Supplementary visual artifacts while using the current inverse-map orientation throughout:

$
X \approx Y A^\top,\qquad A\in\mathbb{R}^{d\times C}.
$

It produces the existing nine class-wise global-feature-importance panels (three datasets by three learners) and `synthetic_heatmaps.png`. The filenames are deliberately unchanged so the current Supplementary LaTeX include paths remain valid.

The class-wise panels retain the six-explainer layout: AIME, HuberAIME, RidgeAIME, HuberRidgeAIME (HRA), LIME, and SHAP. Bars for different classes use separate vertical positions, so coincident class values remain visible.

The controlled synthetic heatmap is equation-consistent: correlation $\rho$ controls the columns of the output design $Y$, because the inverse-map normal system is based on $Y^\top Y$. Outliers are injected into rows of $X$. This replaces the earlier orientation-inconsistent interpretation of input-feature correlation while preserving the role and filename of the heatmap.

The notebook writes raw CSVs, run configuration, environment information, figure provenance, strict validation gates, SHA-256 hashes, and a ZIP package. No failed explainer is silently replaced with zeros.

In [1]:
# %% [user configuration]
USER_OUTPUT_DIR = "./output/supplementary_legacy_corrected"
USER_QUICK_TEST = False
USER_FORCE_RECOMPUTE = True
USER_AUTO_INSTALL = True

# The full public run regenerates all 9 class-wise panels and the synthetic heatmap.
USER_RUN_CLASSWISE_PANELS = True
USER_RUN_SYNTHETIC_HEATMAP = True

# Reproducible resource cap used for HAR before the train/test split.
USER_MAX_HAR_ROWS = 3000

In [2]:
# %% [environment, dependencies, and paths]
import importlib
import importlib.metadata
import io
import json
import math
import os
import platform
import subprocess
import sys
import time
import urllib.request
import warnings
import zipfile
import hashlib
from pathlib import Path


REQUIRED_IMPORTS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "lightgbm": "lightgbm",
    "lime": "lime",
    "shap": "shap",
    "tqdm": "tqdm",
    "requests": "requests",
}


def ensure_dependencies():
    missing = []
    for module_name, package_name in REQUIRED_IMPORTS.items():
        try:
            importlib.import_module(module_name)
        except Exception:
            missing.append(package_name)
    if missing and USER_AUTO_INSTALL:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    elif missing:
        raise ImportError("Missing required packages: " + ", ".join(missing))


ensure_dependencies()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from lime.lime_tabular import LimeTabularExplainer
import shap
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

PIPELINE_VERSION = "supplementary_classwise_synthetic_corrected_2026-08-31_v1"
ORIENTATION = "X ~= Y A^T"
SEED = 42
QUICK_TEST = bool(USER_QUICK_TEST)
FORCE_RECOMPUTE = bool(USER_FORCE_RECOMPUTE)
RUN_CLASSWISE_PANELS = bool(USER_RUN_CLASSWISE_PANELS)
RUN_SYNTHETIC_HEATMAP = bool(USER_RUN_SYNTHETIC_HEATMAP)

OUTDIR = Path(USER_OUTPUT_DIR).expanduser().resolve()
DATADIR = OUTDIR / "data"
FIGDIR = OUTDIR / "figures"
CACHEDIR = OUTDIR / "dataset_cache"
for directory in [OUTDIR, DATADIR, FIGDIR, CACHEDIR]:
    directory.mkdir(parents=True, exist_ok=True)

DATASETS = ["breast_cancer", "credit_approval", "har"]
LEARNERS = ["lgbm", "mlp", "svm"]
AIME_METHODS = ["AIME", "HuberAIME", "RidgeAIME", "HuberRidgeAIME"]
PANEL_METHODS = AIME_METHODS + ["LIME", "SHAP"]
PANEL_FILENAMES = [
    f"panel_{dataset}_{learner}.png"
    for dataset in DATASETS
    for learner in LEARNERS
]
LEGACY_SUPPLEMENTARY_FIGURES = PANEL_FILENAMES + ["synthetic_heatmaps.png"]

RIDGE_LAMBDA = 1e-2
HUBER_DELTA = 1.0
RESIDUAL_MODE = "rms"
MAX_HAR_ROWS = 900 if QUICK_TEST else int(USER_MAX_HAR_ROWS)
TOP_PANEL_FEATURES = 12 if QUICK_TEST else 15

LIME_SAMPLE_N = {
    "breast_cancer": 12 if QUICK_TEST else 128,
    "credit_approval": 12 if QUICK_TEST else 128,
    "har": 8 if QUICK_TEST else 48,
}
LIME_NUM_SAMPLES = {
    "breast_cancer": 300 if QUICK_TEST else 3000,
    "credit_approval": 300 if QUICK_TEST else 3000,
    "har": 250 if QUICK_TEST else 1500,
}
SHAP_SAMPLE_N = {
    "breast_cancer": 8 if QUICK_TEST else 64,
    "credit_approval": 8 if QUICK_TEST else 64,
    "har": 4 if QUICK_TEST else 24,
}
SHAP_BACKGROUND_N = 8 if QUICK_TEST else 48
SHAP_KERNEL_NSAMPLES = 48 if QUICK_TEST else 256

SYNTH_RHOS = [0.0, 0.8] if QUICK_TEST else [0.0, 0.5, 0.8, 0.95]
SYNTH_OUTLIER_RATES = [0.0, 0.2] if QUICK_TEST else [0.0, 0.05, 0.10, 0.20]
SYNTH_REPEATS = 1 if QUICK_TEST else 25
SYNTH_BOOTSTRAPS = 2 if QUICK_TEST else 8
SYNTH_NOISE_TRIALS = 2 if QUICK_TEST else 8
SYNTH_DECOY_TRIALS = 2 if QUICK_TEST else 8
SYNTH_N = 180 if QUICK_TEST else 600
SYNTH_D = 20 if QUICK_TEST else 40
SYNTH_C = 4 if QUICK_TEST else 6
SYNTH_TOP_K = 6 if QUICK_TEST else 10

print("Pipeline:", PIPELINE_VERSION)
print("Orientation:", ORIENTATION)
print("Output:", OUTDIR)
print("Mode:", "quick QA" if QUICK_TEST else "full reproducibility")

Pipeline: supplementary_classwise_synthetic_corrected_2026-08-31_v1
Orientation: X ~= Y A^T
Output: /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/supplementary_legacy_corrected
Mode: full reproducibility


In [3]:
# %% [dataset loading and black-box learners]
def make_unique(names):
    counts = {}
    result = []
    for raw in [str(value) for value in names]:
        count = counts.get(raw, 0)
        result.append(raw if count == 0 else f"{raw}__dup{count}")
        counts[raw] = count + 1
    return result


def download_to_cache(urls, target, timeout=180):
    target = Path(target)
    if target.is_file() and target.stat().st_size > 0:
        return target
    if isinstance(urls, str):
        urls = [urls]
    last_error = None
    for url in urls:
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                payload = response.read()
            target.write_bytes(payload)
            return target
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Dataset download failed for {target.name}: {last_error}")


def load_breast_cancer_data():
    dataset = load_breast_cancer()
    X = pd.DataFrame(dataset.data, columns=make_unique(dataset.feature_names))
    y = pd.Series(dataset.target.astype(int), name="target")
    return X, y, list(map(str, dataset.target_names)), "sklearn/UCI WDBC"


def load_credit_approval_data():
    path = download_to_cache(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/credit-screening/crx.data",
        CACHEDIR / "credit_approval_crx.data",
    )
    raw = pd.read_csv(path, header=None, na_values="?")
    y = raw.iloc[:, -1].map({"+": 1, "-": 0}).astype(int)
    Xraw = raw.iloc[:, :-1].copy()
    Xraw.columns = [f"A{i+1}" for i in range(Xraw.shape[1])]
    categorical = [column for column in Xraw if Xraw[column].dtype == object]
    numeric = [column for column in Xraw if column not in categorical]
    for column in numeric:
        Xraw[column] = pd.to_numeric(Xraw[column], errors="coerce")
        Xraw[column] = Xraw[column].fillna(Xraw[column].median())
    for column in categorical:
        Xraw[column] = Xraw[column].fillna("Unknown").astype(str)
    X = pd.get_dummies(Xraw, columns=categorical, drop_first=True, dtype=float)
    X.columns = make_unique(X.columns)
    return X, y.reset_index(drop=True), ["Denied", "Approved"], "UCI Australian Credit Approval"


def load_har_data():
    path = download_to_cache(
        [
            "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
            "http://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
        ],
        CACHEDIR / "uci_har_dataset.zip",
    )
    with zipfile.ZipFile(path) as archive:
        root = "UCI HAR Dataset/"
        features = pd.read_csv(
            archive.open(root + "features.txt"),
            sep=r"\s+",
            header=None,
            names=["index", "name"],
        )
        labels = pd.read_csv(
            archive.open(root + "activity_labels.txt"),
            sep=r"\s+",
            header=None,
            names=["index", "name"],
        )
        X_train = np.loadtxt(archive.open(root + "train/X_train.txt"))
        X_test = np.loadtxt(archive.open(root + "test/X_test.txt"))
        y_train = np.loadtxt(archive.open(root + "train/y_train.txt")).astype(int)
        y_test = np.loadtxt(archive.open(root + "test/y_test.txt")).astype(int)
    X = pd.DataFrame(
        np.vstack([X_train, X_test]),
        columns=make_unique(features["name"].astype(str)),
    )
    y = pd.Series(np.hstack([y_train, y_test]) - 1, name="target")
    class_names = labels.sort_values("index")["name"].astype(str).tolist()
    return X, y, class_names, "UCI Human Activity Recognition Using Smartphones"


def stratified_cap(X, y, maximum_rows, seed=SEED):
    if maximum_rows is None or len(y) <= maximum_rows:
        return X.reset_index(drop=True), y.reset_index(drop=True)
    splitter = StratifiedShuffleSplit(n_splits=1, train_size=maximum_rows, random_state=seed)
    selected, _ = next(splitter.split(X, y))
    return X.iloc[selected].reset_index(drop=True), y.iloc[selected].reset_index(drop=True)


def load_prepared_dataset(name):
    if name == "breast_cancer":
        Xdf, y, class_names, source = load_breast_cancer_data()
    elif name == "credit_approval":
        Xdf, y, class_names, source = load_credit_approval_data()
    elif name == "har":
        Xdf, y, class_names, source = load_har_data()
        Xdf, y = stratified_cap(Xdf, y, MAX_HAR_ROWS)
    else:
        raise ValueError(name)
    feature_names = list(map(str, Xdf.columns))
    X = StandardScaler().fit_transform(Xdf.to_numpy(float))
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y.to_numpy(int),
        test_size=0.25,
        random_state=SEED,
        stratify=y,
    )
    metadata = {
        "dataset": name,
        "source": source,
        "n_total": int(len(y)),
        "n_train": int(len(y_train)),
        "n_test": int(len(y_test)),
        "d": int(X.shape[1]),
        "classes": int(len(class_names)),
    }
    return X_train, X_test, y_train, y_test, feature_names, class_names, metadata


def build_model(kind):
    if kind == "lgbm":
        return LGBMClassifier(
            random_state=SEED,
            n_estimators=200,
            learning_rate=0.05,
            min_child_samples=20,
            verbosity=-1,
            n_jobs=1,
        )
    if kind == "mlp":
        return MLPClassifier(
            hidden_layer_sizes=(100,),
            max_iter=400,
            early_stopping=True,
            random_state=SEED,
        )
    if kind == "svm":
        return SVC(kernel="rbf", probability=True, random_state=SEED)
    raise ValueError(kind)

In [4]:
# %% [corrected inverse solver and class-wise explainers]
def huber_weights(residual_score, delta):
    residual_score = np.asarray(residual_score, float)
    weights = np.ones_like(residual_score)
    mask = residual_score > delta
    weights[mask] = delta / np.maximum(residual_score[mask], 1e-12)
    return np.clip(weights, 1e-8, 1.0)


def svd_solve(matrix, rhs, rcond=1e-12):
    U, singular_values, Vt = np.linalg.svd(np.asarray(matrix, float), full_matrices=False)
    cutoff = rcond * max(float(singular_values.max()), 1e-300)
    inverse = np.where(singular_values > cutoff, 1.0 / singular_values, 0.0)
    return (Vt.T * inverse) @ (U.T @ np.asarray(rhs, float))


def fit_inverse_map(
    X,
    Y,
    method="AIME",
    delta=HUBER_DELTA,
    ridge_lambda=RIDGE_LAMBDA,
    residual_mode=RESIDUAL_MODE,
    max_iter=100,
    tol=1e-8,
):
    # Fit A(d,C) in X(n,d) ~= Y(n,C) A.T.
    X = np.asarray(X, float)
    Y = np.asarray(Y, float)
    if X.ndim != 2 or Y.ndim != 2 or len(X) != len(Y):
        raise ValueError(f"Expected X(n,d), Y(n,C); got {X.shape}, {Y.shape}")
    n, d = X.shape
    C = Y.shape[1]
    use_huber = method in {"HuberAIME", "HuberRidgeAIME"}
    use_ridge = method in {"RidgeAIME", "HuberRidgeAIME"}
    lam = float(ridge_lambda if use_ridge else 0.0)
    weights = np.ones(n)
    B = np.zeros((C, d), float)
    converged = not use_huber
    iterations = 1
    for iteration in range(max_iter if use_huber else 1):
        gram = Y.T @ (Y * weights[:, None]) + lam * np.eye(C)
        rhs = Y.T @ (X * weights[:, None])
        B_new = svd_solve(gram, rhs)
        residual_norm = np.linalg.norm(X - Y @ B_new, axis=1)
        score = residual_norm / np.sqrt(max(1, d)) if residual_mode == "rms" else residual_norm
        if not use_huber:
            B = B_new
            break
        new_weights = huber_weights(score, delta)
        coefficient_change = np.linalg.norm(B_new - B) / (np.linalg.norm(B) + 1e-12)
        weight_change = np.max(np.abs(new_weights - weights))
        B, weights = B_new, new_weights
        iterations = iteration + 1
        if coefficient_change <= tol and weight_change <= np.sqrt(tol):
            converged = True
            break
    gram = Y.T @ (Y * weights[:, None]) + lam * np.eye(C)
    rhs = Y.T @ (X * weights[:, None])
    B = svd_solve(gram, rhs)
    A = B.T
    if A.shape != (d, C):
        raise AssertionError(f"Orientation gate failed: {A.shape} != {(d, C)}")
    diagnostics = {
        "orientation": ORIENTATION,
        "iterations": int(iterations),
        "converged": bool(converged),
        "mean_weight": float(weights.mean()),
        "fraction_downweighted": float(np.mean(weights < 0.999)),
        "coefficient_frobenius_norm": float(np.linalg.norm(A)),
    }
    return A, diagnostics, weights


def normalize_classwise(A):
    V = np.asarray(A, float).T.copy()
    for class_index in range(V.shape[0]):
        scale = np.abs(V[class_index]).sum()
        if scale > 0:
            V[class_index] /= scale
    return V


def classwise_aime_family(X, model, method):
    Y = np.asarray(model.predict_proba(X), float)
    A, diagnostics, _ = fit_inverse_map(X, Y, method=method)
    return normalize_classwise(A), diagnostics


def classwise_lime(X, model, feature_names, class_names, dataset_name):
    n_classes = model.predict_proba(X[:2]).shape[1]
    d = X.shape[1]
    explainer = LimeTabularExplainer(
        training_data=X,
        feature_names=feature_names,
        class_names=class_names,
        mode="classification",
        discretize_continuous=False,
        kernel_width=0.75 * math.sqrt(d),
        random_state=SEED,
    )
    rng = np.random.default_rng(SEED)
    selected = rng.choice(len(X), size=min(LIME_SAMPLE_N[dataset_name], len(X)), replace=False)
    aggregate = np.zeros((n_classes, d), float)
    class_counts = np.zeros(n_classes, int)
    for row_index in tqdm(selected, desc=f"LIME {dataset_name}", leave=False):
        explanation = explainer.explain_instance(
            X[int(row_index)],
            model.predict_proba,
            labels=list(range(n_classes)),
            num_features=d,
            num_samples=LIME_NUM_SAMPLES[dataset_name],
        )
        mapping = explanation.as_map()
        for class_index in range(n_classes):
            pairs = mapping.get(class_index, [])
            if not pairs:
                raise RuntimeError(f"LIME returned no coefficients for class {class_index}")
            class_counts[class_index] += 1
            for feature_index, weight in pairs:
                if 0 <= int(feature_index) < d:
                    aggregate[class_index, int(feature_index)] += float(weight)
    aggregate /= np.maximum(class_counts[:, None], 1)
    for class_index in range(n_classes):
        scale = np.abs(aggregate[class_index]).sum()
        if scale <= 0:
            raise RuntimeError(f"LIME produced a zero vector for class {class_index}")
        aggregate[class_index] /= scale
    return aggregate, {"sample_n": int(len(selected)), "num_samples": int(LIME_NUM_SAMPLES[dataset_name])}


def shap_values_to_class_feature(values, n_classes, d):
    if isinstance(values, list):
        arrays = [np.asarray(item, float) for item in values]
        if len(arrays) == n_classes:
            return np.stack([array.reshape(array.shape[0], -1)[:, :d].mean(axis=0) for array in arrays])
    array = np.asarray(values, float)
    if array.ndim == 2:
        mean = array[:, :d].mean(axis=0)
        if n_classes == 2:
            return np.vstack([-mean, mean])
        return np.tile(mean[None, :], (n_classes, 1))
    if array.ndim == 3:
        if array.shape[1] == d and array.shape[2] == n_classes:
            return array.mean(axis=0).T
        if array.shape[1] == n_classes and array.shape[2] == d:
            return array.mean(axis=0)
    raise RuntimeError(f"Unsupported SHAP output shape: {array.shape}")


def classwise_shap(X, model, learner, dataset_name):
    n_classes = model.predict_proba(X[:2]).shape[1]
    d = X.shape[1]
    rng = np.random.default_rng(SEED)
    selected = rng.choice(len(X), size=min(SHAP_SAMPLE_N[dataset_name], len(X)), replace=False)
    X_use = X[selected]
    if learner == "lgbm":
        explainer = shap.TreeExplainer(model)
        raw_values = explainer.shap_values(X_use)
        algorithm = "TreeSHAP"
    else:
        background_index = rng.choice(
            len(X), size=min(SHAP_BACKGROUND_N, len(X)), replace=False
        )
        explainer = shap.KernelExplainer(model.predict_proba, X[background_index])
        raw_values = explainer.shap_values(
            X_use,
            nsamples=SHAP_KERNEL_NSAMPLES,
            silent=True,
        )
        algorithm = "KernelSHAP"
    aggregate = shap_values_to_class_feature(raw_values, n_classes, d)
    for class_index in range(n_classes):
        scale = np.abs(aggregate[class_index]).sum()
        if scale <= 0:
            raise RuntimeError(f"SHAP produced a zero vector for class {class_index}")
        aggregate[class_index] /= scale
    return aggregate, {"sample_n": int(len(selected)), "algorithm": algorithm}

In [5]:
# %% [nine class-wise panels]
PANEL_METHOD_LABELS = {
    "AIME": "AIME",
    "HuberAIME": "HuberAIME",
    "RidgeAIME": "RidgeAIME",
    "HuberRidgeAIME": "HuberRidgeAIME",
    "LIME": "LIME",
    "SHAP": "SHAP",
}


def method_csv_path(dataset, learner, method):
    return DATADIR / f"classwise_{dataset}_{learner}_{method}.csv"


def save_classwise_csv(V, dataset, learner, method, feature_names, class_names, metadata):
    rows = []
    for class_index, class_name in enumerate(class_names):
        for feature_index, feature_name in enumerate(feature_names):
            rows.append({
                "pipeline_version": PIPELINE_VERSION,
                "orientation": ORIENTATION,
                "dataset": dataset,
                "learner": learner,
                "method": method,
                "class_index": int(class_index),
                "class_name": str(class_name),
                "feature_index": int(feature_index),
                "feature_name": str(feature_name),
                "signed_weight_l1_normalized": float(V[class_index, feature_index]),
                "method_metadata": json.dumps(metadata, sort_keys=True),
            })
    frame = pd.DataFrame(rows)
    frame.to_csv(method_csv_path(dataset, learner, method), index=False)
    return frame


def load_classwise_csv(dataset, learner, method, n_classes, d):
    path = method_csv_path(dataset, learner, method)
    if not path.is_file():
        return None
    frame = pd.read_csv(path)
    if len(frame) != n_classes * d or set(frame["orientation"]) != {ORIENTATION}:
        return None
    V = np.zeros((n_classes, d), float)
    for row in frame.itertuples(index=False):
        V[int(row.class_index), int(row.feature_index)] = float(row.signed_weight_l1_normalized)
    return V


def shortened_feature_label(label, maximum=38):
    label = str(label)
    return label if len(label) <= maximum else label[: maximum - 1] + "…"


def plot_classwise_panel(V_by_method, dataset, learner, feature_names, class_names):
    n_classes = len(class_names)
    colors = plt.get_cmap("tab10").colors
    fig, axes = plt.subplots(3, 2, figsize=(15.0, 21.0), dpi=220)
    for ax, method in zip(axes.ravel(), PANEL_METHODS):
        V = np.asarray(V_by_method[method], float)
        total_strength = np.abs(V).sum(axis=0)
        selected = np.argsort(-total_strength)[: min(TOP_PANEL_FEATURES, len(feature_names))]
        selected = selected[::-1]
        base_y = np.arange(len(selected), dtype=float)
        bar_height = 0.78 / max(1, n_classes)
        offsets = (np.arange(n_classes) - (n_classes - 1) / 2.0) * bar_height
        for class_index, offset in enumerate(offsets):
            ax.barh(
                base_y + offset,
                V[class_index, selected],
                height=0.92 * bar_height,
                color=colors[class_index % len(colors)],
                label=str(class_names[class_index]),
                alpha=0.92,
            )
        ax.axvline(0, color="0.35", linewidth=0.8, linestyle="--")
        ax.set_yticks(base_y)
        ax.set_yticklabels(
            [shortened_feature_label(feature_names[index]) for index in selected],
            fontsize=8,
        )
        ax.set_xlabel("Signed class-wise importance (L1-normalized)")
        ax.set_title(f"Feature importance by {PANEL_METHOD_LABELS[method]}")
        ax.grid(axis="x", alpha=0.18, linewidth=0.5)
    handles = [
        Line2D([0], [0], color=colors[index % len(colors)], lw=6, label=str(name))
        for index, name in enumerate(class_names)
    ]
    fig.legend(
        handles=handles,
        loc="upper center",
        ncol=min(n_classes, 6),
        frameon=False,
        bbox_to_anchor=(0.5, 0.985),
        title="Class",
    )
    fig.suptitle(
        f"Global feature importance comparison (class-wise)\n({dataset} / {learner})",
        fontsize=15,
        y=0.999,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.965), h_pad=3.0, w_pad=2.0)
    path = FIGDIR / f"panel_{dataset}_{learner}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


CLASSWISE_ERRORS = []
CLASSWISE_AUDIT_ROWS = []


def run_classwise_panels():
    if not RUN_CLASSWISE_PANELS:
        return pd.DataFrame(), []
    all_frames = []
    figure_paths = []
    for dataset in DATASETS:
        prepared = load_prepared_dataset(dataset)
        X_train, X_test, y_train, y_test, feature_names, class_names, dataset_meta = prepared
        for learner in LEARNERS:
            print(f"\n[class-wise] {dataset} / {learner}")
            model = build_model(learner)
            started = time.perf_counter()
            model.fit(X_train, y_train)
            fit_seconds = time.perf_counter() - started
            accuracy = float(accuracy_score(y_test, model.predict(X_test)))
            n_classes = model.predict_proba(X_train[:2]).shape[1]
            if n_classes != len(class_names):
                class_names = [str(value) for value in model.classes_]
            V_by_method = {}
            for method in PANEL_METHODS:
                cached = None if FORCE_RECOMPUTE else load_classwise_csv(
                    dataset, learner, method, n_classes, X_train.shape[1]
                )
                if cached is not None:
                    V = cached
                    metadata = {"loaded_from_cache": True}
                    all_frames.append(pd.read_csv(method_csv_path(dataset, learner, method)))
                else:
                    try:
                        method_started = time.perf_counter()
                        if method in AIME_METHODS:
                            V, metadata = classwise_aime_family(X_train, model, method)
                        elif method == "LIME":
                            V, metadata = classwise_lime(
                                X_train, model, feature_names, class_names, dataset
                            )
                        else:
                            V, metadata = classwise_shap(X_train, model, learner, dataset)
                        metadata = dict(metadata)
                        metadata["elapsed_seconds"] = float(time.perf_counter() - method_started)
                        frame = save_classwise_csv(
                            V, dataset, learner, method, feature_names, class_names, metadata
                        )
                        all_frames.append(frame)
                    except Exception as exc:
                        CLASSWISE_ERRORS.append({
                            "dataset": dataset,
                            "learner": learner,
                            "method": method,
                            "error": repr(exc),
                        })
                        raise
                if not np.isfinite(V).all() or np.max(np.abs(V)) <= 0:
                    raise RuntimeError(f"Invalid class-wise values: {dataset}/{learner}/{method}")
                V_by_method[method] = V
            figure_paths.append(
                plot_classwise_panel(
                    V_by_method, dataset, learner, feature_names, class_names
                )
            )
            CLASSWISE_AUDIT_ROWS.append({
                **dataset_meta,
                "learner": learner,
                "test_accuracy": accuracy,
                "model_fit_seconds": fit_seconds,
                "orientation": ORIENTATION,
            })
    merged = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
    if len(merged):
        merged.to_csv(DATADIR / "classwise_all_methods_raw.csv", index=False)
    pd.DataFrame(CLASSWISE_AUDIT_ROWS).to_csv(
        DATADIR / "classwise_dataset_model_audit.csv", index=False
    )
    return merged, figure_paths

In [6]:
# %% [equation-consistent controlled synthetic heatmap]
def cosine_safe(first, second):
    a = np.asarray(first, float).ravel()
    b = np.asarray(second, float).ravel()
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0


def topk_set_from_operator(A, k):
    strength = np.linalg.norm(np.asarray(A, float), axis=1)
    k = min(int(k), len(strength))
    return set(np.argsort(-strength)[:k].tolist())


def jaccard(first, second):
    union = first | second
    return float(len(first & second) / len(union)) if union else 1.0


def make_synthetic_inverse_problem(rho, outlier_rate, seed):
    rng = np.random.default_rng(seed)
    index = np.arange(SYNTH_C)
    covariance = float(rho) ** np.abs(np.subtract.outer(index, index))
    Y = rng.multivariate_normal(np.zeros(SYNTH_C), covariance, size=SYNTH_N)
    Y = StandardScaler().fit_transform(Y)
    A_true = np.zeros((SYNTH_D, SYNTH_C), float)
    active = min(SYNTH_TOP_K, SYNTH_D)
    A_true[:active] = rng.normal(0, 1, size=(active, SYNTH_C))
    A_true[:active] *= np.linspace(1.5, 0.5, active)[:, None]
    X_clean = Y @ A_true.T + rng.normal(0, 0.10, size=(SYNTH_N, SYNTH_D))
    X_observed = X_clean.copy()
    mask = np.zeros(SYNTH_N, bool)
    contaminated_rows = int(round(float(outlier_rate) * SYNTH_N))
    if contaminated_rows > 0:
        selected = rng.choice(SYNTH_N, size=contaminated_rows, replace=False)
        mask[selected] = True
        X_observed[selected] += rng.standard_t(
            df=2.0, size=(contaminated_rows, SYNTH_D)
        ) * 4.0
    return X_observed, X_clean, Y, A_true, mask


def bootstrap_stability_metric(X, Y, method, A_reference, seed):
    rng = np.random.default_rng(seed)
    reference_top = topk_set_from_operator(A_reference, SYNTH_TOP_K)
    values = []
    for _ in range(SYNTH_BOOTSTRAPS):
        indices = rng.choice(len(X), size=len(X), replace=True)
        A, _, _ = fit_inverse_map(X[indices], Y[indices], method=method)
        values.append(jaccard(reference_top, topk_set_from_operator(A, SYNTH_TOP_K)))
    return float(np.mean(values))


def noise_robustness_metric(X, Y, method, A_reference, seed):
    rng = np.random.default_rng(seed)
    reference_top = topk_set_from_operator(A_reference, SYNTH_TOP_K)
    feature_scale = np.std(X, axis=0, keepdims=True)
    feature_scale = np.where(feature_scale > 1e-12, feature_scale, 1.0)
    values = []
    for _ in range(SYNTH_NOISE_TRIALS):
        perturbed = X + rng.normal(0, 0.05 * feature_scale, size=X.shape)
        A, _, _ = fit_inverse_map(perturbed, Y, method=method)
        values.append(jaccard(reference_top, topk_set_from_operator(A, SYNTH_TOP_K)))
    return float(np.mean(values))


def decoy_resistance_metric(X, Y, method, seed):
    rng = np.random.default_rng(seed)
    values = []
    d = X.shape[1]
    for _ in range(SYNTH_DECOY_TRIALS):
        decoy = rng.normal(0, 1, size=(len(X), 1))
        extended = np.hstack([X, decoy])
        A, _, _ = fit_inverse_map(extended, Y, method=method)
        strength = np.linalg.norm(A, axis=1)
        top = topk_set_from_operator(A, SYNTH_TOP_K)
        top_infiltration = 1.0 if d in top else 0.0
        mass = float(strength[d] / (strength.sum() + 1e-12))
        values.append(1.0 - 0.5 * (top_infiltration + mass))
    return float(np.mean(values))


def run_synthetic_grid():
    path = DATADIR / "synthetic_grid_raw.csv"
    if path.is_file() and not FORCE_RECOMPUTE:
        cached = pd.read_csv(path)
        if set(cached.get("pipeline_version", [])) == {PIPELINE_VERSION}:
            return cached
    rows = []
    total = len(SYNTH_RHOS) * len(SYNTH_OUTLIER_RATES) * SYNTH_REPEATS
    progress = tqdm(total=total, desc="Synthetic cells")
    for rho_index, rho in enumerate(SYNTH_RHOS):
        for outlier_index, outlier_rate in enumerate(SYNTH_OUTLIER_RATES):
            for repeat in range(SYNTH_REPEATS):
                problem_seed = SEED + repeat + 1000 * rho_index + 100000 * outlier_index
                X, X_clean, Y, A_true, outlier_mask = make_synthetic_inverse_problem(
                    rho, outlier_rate, problem_seed
                )
                for method in AIME_METHODS:
                    A, diagnostics, weights = fit_inverse_map(X, Y, method=method)
                    rows.append({
                        "pipeline_version": PIPELINE_VERSION,
                        "orientation": ORIENTATION,
                        "rho_output_design": float(rho),
                        "outlier_rate": float(outlier_rate),
                        "repeat": int(repeat),
                        "method": method,
                        "stability": bootstrap_stability_metric(
                            X, Y, method, A, problem_seed + 101
                        ),
                        "noise": noise_robustness_metric(
                            X, Y, method, A, problem_seed + 202
                        ),
                        "decoy": decoy_resistance_metric(
                            X, Y, method, problem_seed + 303
                        ),
                        "operator_cosine_to_truth": cosine_safe(A, A_true),
                        "mean_huber_weight": float(weights.mean()),
                        "outlier_mean_weight": float(weights[outlier_mask].mean()) if outlier_mask.any() else np.nan,
                        "clean_mean_weight": float(weights[~outlier_mask].mean()),
                        "converged": bool(diagnostics["converged"]),
                    })
                progress.update(1)
    progress.close()
    frame = pd.DataFrame(rows)
    frame.to_csv(path, index=False)
    return frame


def plot_synthetic_heatmap(frame):
    metrics = [
        ("stability", "Bootstrap stability"),
        ("noise", "Noise robustness"),
        ("decoy", "Decoy resistance"),
    ]
    fig, axes = plt.subplots(3, 4, figsize=(18.5, 11.0), dpi=220)
    image_artist = None
    for row_index, (metric, metric_label) in enumerate(metrics):
        for column_index, method in enumerate(AIME_METHODS):
            ax = axes[row_index, column_index]
            subset = frame[frame["method"] == method]
            matrix = (
                subset.groupby(["rho_output_design", "outlier_rate"])[metric]
                .mean()
                .unstack("outlier_rate")
                .reindex(index=SYNTH_RHOS, columns=SYNTH_OUTLIER_RATES)
            )
            values = matrix.to_numpy(float)
            image_artist = ax.imshow(
                values,
                vmin=0,
                vmax=1,
                cmap="viridis",
                aspect="auto",
                origin="lower",
            )
            for y_index in range(values.shape[0]):
                for x_index in range(values.shape[1]):
                    value = values[y_index, x_index]
                    text_color = "white" if value < 0.55 else "black"
                    ax.text(x_index, y_index, f"{value:.2f}", ha="center", va="center", fontsize=7, color=text_color)
            ax.set_xticks(np.arange(len(SYNTH_OUTLIER_RATES)))
            ax.set_xticklabels([f"{value:.2f}" for value in SYNTH_OUTLIER_RATES])
            ax.set_yticks(np.arange(len(SYNTH_RHOS)))
            ax.set_yticklabels([f"{value:.2f}" for value in SYNTH_RHOS])
            ax.set_xlabel("Outlier rate π")
            if column_index == 0:
                ax.set_ylabel("Output-design correlation ρ")
            ax.set_title(f"{method} — {metric_label}")
    fig.subplots_adjust(left=0.06, right=0.88, bottom=0.07, top=0.93, wspace=0.30, hspace=0.36)
    colorbar_axis = fig.add_axes([0.905, 0.16, 0.014, 0.68])
    colorbar = fig.colorbar(image_artist, cax=colorbar_axis)
    colorbar.set_label("Mean score (higher is better)")
    fig.suptitle(
        "Controlled inverse-map grid: output-design collinearity and row outliers",
        fontsize=15,
        y=0.985,
    )
    path = FIGDIR / "synthetic_heatmaps.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path

In [7]:
# %% [validation, provenance, and reproducibility package]
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def run_orientation_gates():
    rng = np.random.default_rng(812)
    Y = rng.normal(size=(90, 4))
    X = rng.normal(size=(90, 11))
    A, _, _ = fit_inverse_map(X, Y, method="AIME")
    expected = (np.linalg.pinv(Y) @ X).T
    ridge, _, _ = fit_inverse_map(X, Y, method="RidgeAIME")
    large_delta_hra, _, _ = fit_inverse_map(
        X, Y, method="HuberRidgeAIME", delta=1e12
    )
    X_out = X.copy()
    X_out[:9] += 25.0
    _, _, weights = fit_inverse_map(X_out, Y, method="HuberRidgeAIME", delta=1.0)
    return {
        "orientation_shape_d_by_C": A.shape == (X.shape[1], Y.shape[1]),
        "aime_matches_pinv_Y_times_X": bool(np.allclose(A, expected, rtol=1e-8, atol=1e-10)),
        "large_delta_hra_matches_ridge": bool(np.allclose(large_delta_hra, ridge, rtol=1e-7, atol=1e-9)),
        "huber_downweights_injected_rows": bool(weights[:9].mean() < weights[9:].mean()),
    }


def write_supporting_artifacts(classwise_frame, synthetic_frame):
    configuration = {
        "pipeline_version": PIPELINE_VERSION,
        "orientation": ORIENTATION,
        "quick_test": QUICK_TEST,
        "force_recompute": FORCE_RECOMPUTE,
        "datasets": DATASETS,
        "learners": LEARNERS,
        "panel_methods": PANEL_METHODS,
        "panel_filenames": PANEL_FILENAMES,
        "synthetic_heatmap_filename": "synthetic_heatmaps.png",
        "ridge_lambda": RIDGE_LAMBDA,
        "huber_delta": HUBER_DELTA,
        "residual_mode": RESIDUAL_MODE,
        "max_har_rows": MAX_HAR_ROWS,
        "lime_sample_n": LIME_SAMPLE_N,
        "lime_num_samples": LIME_NUM_SAMPLES,
        "shap_sample_n": SHAP_SAMPLE_N,
        "shap_background_n": SHAP_BACKGROUND_N,
        "shap_kernel_nsamples": SHAP_KERNEL_NSAMPLES,
        "synthetic_rhos_output_design": SYNTH_RHOS,
        "synthetic_outlier_rates": SYNTH_OUTLIER_RATES,
        "synthetic_repeats": SYNTH_REPEATS,
        "synthetic_bootstraps": SYNTH_BOOTSTRAPS,
        "synthetic_noise_trials": SYNTH_NOISE_TRIALS,
        "synthetic_decoy_trials": SYNTH_DECOY_TRIALS,
        "synthetic_n": SYNTH_N,
        "synthetic_d": SYNTH_D,
        "synthetic_C": SYNTH_C,
    }
    (OUTDIR / "supplementary_legacy_corrected_run_configuration.json").write_text(
        json.dumps(configuration, indent=2), encoding="utf-8"
    )

    package_names = list(REQUIRED_IMPORTS.values())
    versions = {}
    for package in package_names:
        try:
            versions[package] = importlib.metadata.version(package)
        except Exception:
            versions[package] = "unknown"
    environment = {
        "python": sys.version,
        "platform": platform.platform(),
        "packages": versions,
    }
    (OUTDIR / "supplementary_legacy_corrected_environment_manifest.json").write_text(
        json.dumps(environment, indent=2), encoding="utf-8"
    )

    provenance_rows = []
    for filename in PANEL_FILENAMES:
        stem = filename.removeprefix("panel_").removesuffix(".png")
        dataset, learner = stem.rsplit("_", 1)
        provenance_rows.append({
            "artifact": filename,
            "source": f"classwise_{dataset}_{learner}_<method>.csv",
            "scope": "Corrected class-wise signed global feature importance; six explainers",
            "orientation": ORIENTATION,
        })
    provenance_rows.append({
        "artifact": "synthetic_heatmaps.png",
        "source": "synthetic_grid_raw.csv",
        "scope": "Controlled output-design collinearity and row-outlier experiment",
        "orientation": ORIENTATION,
    })
    pd.DataFrame(provenance_rows).to_csv(
        DATADIR / "supplementary_legacy_corrected_figure_provenance.csv", index=False
    )


def validate_outputs(classwise_frame, synthetic_frame):
    checks = run_orientation_gates()
    checks["no_classwise_errors"] = len(CLASSWISE_ERRORS) == 0
    checks["all_ten_figure_files_exist"] = all(
        (FIGDIR / filename).is_file() and (FIGDIR / filename).stat().st_size > 1000
        for filename in LEGACY_SUPPLEMENTARY_FIGURES
    ) if RUN_CLASSWISE_PANELS and RUN_SYNTHETIC_HEATMAP else True
    checks["nine_panel_files_exist"] = all(
        (FIGDIR / filename).is_file() for filename in PANEL_FILENAMES
    ) if RUN_CLASSWISE_PANELS else True
    if RUN_CLASSWISE_PANELS:
        observed = {
            (row.dataset, row.learner, row.method)
            for row in classwise_frame[["dataset", "learner", "method"]].drop_duplicates().itertuples(index=False)
        } if len(classwise_frame) else {
            (dataset, learner, method)
            for dataset in DATASETS for learner in LEARNERS for method in PANEL_METHODS
            if method_csv_path(dataset, learner, method).is_file()
        }
        expected = {
            (dataset, learner, method)
            for dataset in DATASETS for learner in LEARNERS for method in PANEL_METHODS
        }
        checks["all_54_classwise_method_cells"] = observed == expected
    else:
        checks["all_54_classwise_method_cells"] = True
    if RUN_SYNTHETIC_HEATMAP:
        expected_rows = (
            len(SYNTH_RHOS)
            * len(SYNTH_OUTLIER_RATES)
            * SYNTH_REPEATS
            * len(AIME_METHODS)
        )
        checks["synthetic_expected_row_count"] = len(synthetic_frame) == expected_rows
        metric_values = synthetic_frame[["stability", "noise", "decoy"]].to_numpy(float)
        checks["synthetic_metrics_finite_and_bounded"] = bool(
            np.isfinite(metric_values).all()
            and (metric_values >= 0).all()
            and (metric_values <= 1).all()
        )
        checks["synthetic_orientation_recorded"] = set(synthetic_frame["orientation"]) == {ORIENTATION}
    else:
        checks["synthetic_expected_row_count"] = True
        checks["synthetic_metrics_finite_and_bounded"] = True
        checks["synthetic_orientation_recorded"] = True

    failed = [name for name, passed in checks.items() if not bool(passed)]
    report = {
        "pipeline_version": PIPELINE_VERSION,
        "orientation": ORIENTATION,
        "checks": checks,
        "failed_checks": failed,
        "passed": len(failed) == 0,
        "classwise_errors": CLASSWISE_ERRORS,
    }
    (OUTDIR / "supplementary_legacy_corrected_validation_report.json").write_text(
        json.dumps(report, indent=2), encoding="utf-8"
    )
    if failed:
        raise AssertionError("Validation failed: " + ", ".join(failed))
    return report


def build_manifest_and_zip():
    manifest_path = OUTDIR / "supplementary_legacy_corrected_artifact_manifest_sha256.csv"
    archive_path = OUTDIR / "HuberRidgeAIME_Supplementary_Classwise_and_Synthetic_Visuals_CORRECTED_outputs.zip"
    artifacts = [
        path for path in OUTDIR.rglob("*")
        if (
            path.is_file()
            and path not in {manifest_path, archive_path}
            and CACHEDIR not in path.parents
        )
    ]
    rows = [
        {
            "relative_path": str(path.relative_to(OUTDIR)),
            "bytes": int(path.stat().st_size),
            "sha256": sha256_file(path),
        }
        for path in sorted(artifacts)
    ]
    pd.DataFrame(rows).to_csv(manifest_path, index=False)
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(artifacts + [manifest_path]):
            archive.write(path, arcname=str(path.relative_to(OUTDIR)))
    return manifest_path, archive_path

In [8]:
# %% [run all requested experiments]
CLASSWISE_RAW = pd.DataFrame()
CLASSWISE_FIGURE_PATHS = []
SYNTHETIC_RAW = pd.DataFrame()
SYNTHETIC_FIGURE_PATH = None

if RUN_CLASSWISE_PANELS:
    CLASSWISE_RAW, CLASSWISE_FIGURE_PATHS = run_classwise_panels()

if RUN_SYNTHETIC_HEATMAP:
    SYNTHETIC_RAW = run_synthetic_grid()
    SYNTHETIC_FIGURE_PATH = plot_synthetic_heatmap(SYNTHETIC_RAW)

write_supporting_artifacts(CLASSWISE_RAW, SYNTHETIC_RAW)
VALIDATION_REPORT = validate_outputs(CLASSWISE_RAW, SYNTHETIC_RAW)
MANIFEST_PATH, ZIP_PATH = build_manifest_and_zip()

print("\nValidation passed:", VALIDATION_REPORT["passed"])
print("Figures:")
for filename in LEGACY_SUPPLEMENTARY_FIGURES:
    print(" -", FIGDIR / filename)
print("ZIP:", ZIP_PATH)


[class-wise] breast_cancer / lgbm


LIME breast_cancer:   0%|          | 0/128 [00:00<?, ?it/s]


[class-wise] breast_cancer / mlp


LIME breast_cancer:   0%|          | 0/128 [00:00<?, ?it/s]


[class-wise] breast_cancer / svm


...

LIME har:   0%|          | 0/48 [00:00<?, ?it/s]


[class-wise] har / svm


LIME har:   0%|          | 0/48 [00:00<?, ?it/s]

Synthetic cells:   0%|          | 0/400 [00:00<?, ?it/s]


Validation passed: True
Figures:
 - /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/supplementary_legacy_corrected/figures/panel_breast_cancer_lgbm.png
 - /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/supplementary_legacy_corrected/figures/panel_breast_cancer_mlp.png
 - /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/supplementary_legacy_corrected/figures/panel_breast_cancer_svm.png
 - /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/supplementary_legacy_corrected/figures/panel_credit_approval_lgbm.png
 - /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/supplementary_legacy_corrected/figures/panel_credit_approval_mlp.png
 - /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/supplementary_legacy_corrected/figures/panel_credit_approval_svm.png
 - /Users/takafumi/Documents/Python/HuberRidgeAIME/20260831/notebooks/output/supplementary_leg